# Задание 3. Выручка и маржинальность по брендам

Сравнение недель 14 и 15: динамика выручки, валовой прибыли и маржинальности, а также основные изменения по брендам.

## Подход

Общую маржинальность рассчитываем как сумму валовой прибыли, делённую на сумму выручки. Простое среднее маржинальностей брендов не учитывает разницу в их объёмах продаж.

In [1]:
import pandas as pd

file = 'Тестовое_задание_Data_аналитик_ЦО_2025_2(1).xlsx'
brands = pd.read_excel(file, sheet_name='Данные для задачи 3')

print('Строк:', brands.shape[0])
brands.head()

Строк: 104


,Бренд AX 1,Неделя,Выручка,Маржинальность
0,Shiseido,14,1.353686e+09,0.167824
1,Clarins,14,1.329934e+09,0.091348
2,Shiseido,15,1.267702e+09,0.169212
3,Clarins,15,9.747119e+08,0.145904
4,Clinique,15,4.982012e+08,0.140136


## 1. Валовая прибыль

В исходном листе маржинальность дана в долях. Рассчитываем валовую прибыль как выручку, умноженную на маржинальность.

In [2]:
brands['Валовая прибыль'] = brands['Выручка'] * brands['Маржинальность']

brands[['Бренд AX 1', 'Неделя', 'Выручка', 'Маржинальность',
        'Валовая прибыль']].head(10)

,Бренд AX 1,Неделя,Выручка,Маржинальность,Валовая прибыль
0,Shiseido,14,1.353686e+09,0.167824,2.271804e+08
1,Clarins,14,1.329934e+09,0.091348,1.214863e+08
2,Shiseido,15,1.267702e+09,0.169212,2.145103e+08
3,Clarins,15,9.747119e+08,0.145904,1.422140e+08
4,Clinique,15,4.982012e+08,0.140136,6.981598e+07
5,Clinique,14,4.853968e+08,0.169744,8.239300e+07
6,Payot,15,3.681262e+08,0.209797,7.723190e+07
7,Estee Lauder,14,3.287410e+08,0.108340,3.561569e+07
8,Estee Lauder,15,3.259306e+08,0.104683,3.411935e+07
9,BABOR,14,3.092823e+08,0.147721,4.568743e+07


## 2. Результаты по неделям

Агрегируем выручку и валовую прибыль, затем рассчитываем общую маржинальность по каждой неделе.

In [3]:
weekly = brands.groupby('Неделя').agg(Выручка=('Выручка', 'sum'), Валовая_прибыль=('Валовая прибыль', 'sum')).reset_index()

weekly['Маржинальность %'] = (
    100 * weekly['Валовая_прибыль'] / weekly['Выручка']
)
weekly

,Неделя,Выручка,Валовая_прибыль,Маржинальность %
0,14,8.464189e+09,1.237988e+09,14.626183
1,15,8.207908e+09,1.268186e+09,15.450787


## 3. Динамика по брендам

Соединяем данные двух недель. Бренды, присутствующие только в одной неделе, сохранила.

In [4]:
week14 = brands[brands['Неделя'] == 14][
    ['Бренд AX 1', 'Выручка', 'Маржинальность', 'Валовая прибыль']
].rename(columns={
    'Выручка': 'Выручка 14',
    'Маржинальность': 'Маржинальность 14',
    'Валовая прибыль': 'Прибыль 14'
})

week15 = brands[brands['Неделя'] == 15][
    ['Бренд AX 1', 'Выручка', 'Маржинальность', 'Валовая прибыль']
].rename(columns={
    'Выручка': 'Выручка 15',
    'Маржинальность': 'Маржинальность 15',
    'Валовая прибыль': 'Прибыль 15'
})

by_brand = pd.merge(week14, week15, on='Бренд AX 1', how='outer')
by_brand.head()

,Бренд AX 1,Выручка 14,Маржинальность 14,Прибыль 14,Выручка 15,Маржинальность 15,Прибыль 15
0,Ahava,4.441065e+07,0.124377,5.523678e+06,7.315324e+07,0.081975,5.996762e+06
1,Anne Semonin,3.267357e+07,0.202139,6.604613e+06,1.924395e+07,0.201560,3.878819e+06
2,BABOR,3.092823e+08,0.147721,4.568743e+07,2.908420e+08,0.147962,4.303360e+07
3,Bellefontaine,1.783494e+07,0.107635,1.919656e+06,NaN,NaN,NaN
4,Biothal,NaN,NaN,NaN,1.794943e+07,0.141663,2.542771e+06


## 4. Изменение показателей

Выручку сравниваем в денежных единицах и процентах, а маржинальность — в процентных пунктах.

In [5]:
by_brand['Выручка изменение'] = by_brand['Выручка 15'] - by_brand['Выручка 14']
by_brand['Выручка изменение %'] = (
    100 * by_brand['Выручка изменение'] / by_brand['Выручка 14']
)
by_brand['Прибыль изменение'] = by_brand['Прибыль 15'] - by_brand['Прибыль 14']
by_brand['Маржинальность изменение п.п.'] = (
    100 * (by_brand['Маржинальность 15'] - by_brand['Маржинальность 14'])
)

by_brand.sort_values('Прибыль изменение')[
    ['Бренд AX 1', 'Выручка изменение %', 'Прибыль изменение',
     'Маржинальность изменение п.п.']
].head(10)

,Бренд AX 1,Выручка изменение %,Прибыль изменение,Маржинальность изменение п.п.
47,Shiseido,-6.351775,-1.267008e+07,0.138826
15,Clinique,2.637922,-1.257701e+07,-2.960747
48,Sisley,-31.432426,-1.233543e+07,1.156808
46,SENSAI,-22.166399,-5.581152e+06,-0.240098
17,Dr Barbara Sturm,-34.355312,-3.984443e+06,-2.858961
7,CHANEL,-8.189139,-3.979465e+06,-1.048272
30,Helena Rubinstein,-25.738392,-3.374353e+06,-4.939537
22,FILORGA,-26.951999,-3.171821e+06,0.503780
1,Anne Semonin,-41.102384,-2.725794e+06,-0.057892
2,BABOR,-5.962305,-2.653832e+06,0.024134


In [6]:
by_brand.sort_values('Прибыль изменение', ascending=False)[
    ['Бренд AX 1', 'Выручка изменение %', 'Прибыль изменение',
     'Маржинальность изменение п.п.']
].head(10)

,Бренд AX 1,Выручка изменение %,Прибыль изменение,Маржинальность изменение п.п.
13,Clarins,-26.709771,2.072776e+07,5.455609
43,Payot,38.480411,1.966856e+07,-0.674240
53,Valmont,38.783531,1.435105e+07,1.028176
34,La Mer,20.178903,8.121260e+06,0.059443
50,St.Barth,51.751260,6.431818e+06,-0.272944
35,La Prairie,5.111131,6.045029e+06,2.133302
33,Kiehls,11.287834,3.870925e+06,-0.086825
41,OK Beauty,37.389651,3.678496e+06,0.277952
52,Thalgo,21.583968,3.655179e+06,2.195402
10,Caudalie,25.272464,3.388019e+06,-2.304126


## 5. Основные факторы

На общую маржинальность влияют маржинальность отдельных брендов и структура выручки. 

Для определения возможных причин изменения смотрим прежде всего на крупные бренды и существенные изменения прибыли.

In [7]:
by_brand.sort_values('Выручка 14', ascending=False)[
    ['Бренд AX 1', 'Выручка 14', 'Выручка 15',
     'Маржинальность 14', 'Маржинальность 15', 'Прибыль изменение']
].head(12)

,Бренд AX 1,Выручка 14,Выручка 15,Маржинальность 14,Маржинальность 15,Прибыль изменение
47,Shiseido,1.353686e+09,1.267702e+09,0.167824,0.169212,-1.267008e+07
13,Clarins,1.329934e+09,9.747119e+08,0.091348,0.145904,2.072776e+07
15,Clinique,4.853968e+08,4.982012e+08,0.169744,0.140136,-1.257701e+07
21,Estee Lauder,3.287410e+08,3.259306e+08,0.108340,0.104683,-1.496339e+06
2,BABOR,3.092823e+08,2.908420e+08,0.147721,0.147962,-2.653832e+06
48,Sisley,3.013878e+08,2.066543e+08,0.155447,0.167015,-1.233543e+07
43,Payot,2.658327e+08,3.681262e+08,0.216540,0.209797,1.966856e+07
37,Lancome,2.413461e+08,2.042886e+08,0.102322,0.127443,1.340172e+06
7,CHANEL,2.367033e+08,2.173193e+08,0.087772,0.077289,-3.979465e+06
34,La Mer,2.280139e+08,2.740246e+08,0.172968,0.173562,8.121260e+06


## Выводы

Выручка снизилась с 8,464 до 8,208 млрд денежных единиц (**−3,03%**), а расчётная валовая прибыль увеличилась с 1,238 до 1,268 млрд (**+2,44%**). Общая маржинальность выросла с **14,63% до 15,45%** (**+0,82 п.п.**).

Основные изменения:
- **Clarins:** выручка −26,7%, маржинальность +5,46 п.п., прибыль +20,7 млн.
- **Payot:** выручка +38,5%, прибыль +19,7 млн.
- **Clinique:** выручка +2,6%, но маржинальность −2,96 п.п. и прибыль −12,6 млн.
- **Shiseido:** снижение валовой прибыли примерно на 12,7 млн.

**Предлагаемое действие:** проверить скидки, закупочные условия и состав проданных товаров Clinique.

Дополнительно разобраться с падением продаж Clarins. Необходимо больше данных  по товарам,  скидкам  и т.д.
